# Checkpoint 46 — Retention Policy Analysis

This notebook reviews the validation-selected, once-only test evaluation created by `src/optimize_retention_policy.py`. The policy is frozen before the 2025 test target is accessed.

In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

processed = project_root / "data" / "processed"
comparison = pd.read_csv(processed / "retention_policy_comparison.csv")
decision = pd.read_csv(processed / "retention_policy_decision.csv")
model_metrics = pd.read_csv(processed / "retention_final_test_model_metrics.csv")
sensitivity = pd.read_csv(processed / "retention_policy_sensitivity.csv")
current_summary = pd.read_csv(processed / "current_retention_policy_summary.csv")
current_groups = pd.read_csv(processed / "current_retention_policy_group_summary.csv")
validation = pd.read_csv(processed / "retention_policy_validation.csv")

## 1. Policy comparison

The fixed 0.50 threshold, top-k rules, cost threshold, and budget-constrained policy are shown together. `Outcome-aligned net value` uses observed attrition outcomes but still assumes the intervention success rate; it is not observed profit.

In [ ]:
comparison[[
    "evaluation_period",
    "policy",
    "selected_count",
    "selected_positive_cases",
    "precision",
    "capture_rate",
    "intervention_spend_usd",
    "expected_net_value_usd",
    "outcome_aligned_net_value_usd",
]]

## 2. Frozen final decision

The SHA-256 value records the exact policy configuration that existed before the final test evaluation.

In [ ]:
decision.T

## 3. Final model metrics

These metrics come from the previously reserved 2025 test snapshot. The policy must not be changed in response to them.

In [ ]:
model_metrics

## 4. Cost sensitivity

The selected employee set is revalued across all 27 cost scenarios without changing the test policy.

In [ ]:
selected_sensitivity = sensitivity.loc[
    sensitivity["policy_key"].eq("budget_expected_value")
    & sensitivity["evaluation_period"].eq("2025 final test")
]

selected_sensitivity[[
    "scenario_id",
    "expected_net_value_usd",
    "outcome_aligned_net_value_usd",
]].sort_values("outcome_aligned_net_value_usd")

## 5. Current synthetic plan

The current plan is for human review only. Financial value is not employee value, and the scores must not trigger automatic employment actions.

In [ ]:
current_summary

In [ ]:
current_groups.sort_values("selection_rate", ascending=False)

## 6. Validation

Every row must be `PASS`, including policy freeze, one-time test access, budget enforcement, employee-disjoint calibration, scenario coverage, and the human-review restriction.

In [ ]:
validation